# Exercise: Encoding high-cardinality features

In this notebook we will **demonstrate the issue with one-hot encoding high-cardinality categorical features**.

You already saw in `lab_encode` that one-hot encoding can dramatically increase the number of columns and memory usage for datasets with many categories (like Ames Housing). Here, you will **reproduce this effect yourself step-by-step**:

1. **Load** the Ames Housing data and take a quick look at it.
2. **Identify** the categorical features.
3. **Measure** how much memory the categorical part of the data uses *before* encoding.
4. **Apply** one-hot encoding to those categorical features.
5. **Measure and compare** the memory usage *after* encoding.

As you go, pause after each step and write down what you **expect** to happen (e.g., *"I expect the number of columns to increase by about X"*), then check against the actual results.

In [ ]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.datasets import fetch_openml

ames_housing = fetch_openml(data_id=41211, as_frame=True)
data = ames_housing.data
target = ames_housing.target

data

/home/halgoz/work/udacity-c6/c6/content/modules/.venv/lib/python3.12/site-packages/sklearn/datasets/_openml.py:1035: UserWarning: Version 1 of dataset ames-housing is inactive, meaning that issues have been found in the dataset. Try using a newer version from this URL: https://openml.org/data/v1/download/20649135/ames-housing.arff
  warn(


,MS_SubClass,MS_Zoning,Lot_Frontage,Lot_Area,Street,Alley,Lot_Shape,Land_Contour,Utilities,Lot_Config,...,Pool_QC,Fence,Misc_Feature,Misc_Val,Mo_Sold,Year_Sold,Sale_Type,Sale_Condition,Longitude,Latitude
0,One_Story_1946_and_Newer_All_Styles,Residential_Low_Density,141,31770,Pave,No_Alley_Access,Slightly_Irregular,Lvl,AllPub,Corner,...,No_Pool,No_Fence,None,0,5,2010,WD,Normal,-93.619754,42.054035
1,One_Story_1946_and_Newer_All_Styles,Residential_High_Density,80,11622,Pave,No_Alley_Access,Regular,Lvl,AllPub,Inside,...,No_Pool,Minimum_Privacy,None,0,6,2010,WD,Normal,-93.619756,42.053014
2,One_Story_1946_and_Newer_All_Styles,Residential_Low_Density,81,14267,Pave,No_Alley_Access,Slightly_Irregular,Lvl,AllPub,Corner,...,No_Pool,No_Fence,Gar2,12500,6,2010,WD,Normal,-93.619387,42.052659
3,One_Story_1946_and_Newer_All_Styles,Residential_Low_Density,93,11160,Pave,No_Alley_Access,Regular,Lvl,AllPub,Corner,...,No_Pool,No_Fence,None,0,4,2010,WD,Normal,-93.617320,42.051245
4,Two_Story_1946_and_Newer,Residential_Low_Density,74,13830,Pave,No_Alley_Access,Slightly_Irregular,Lvl,AllPub,Inside,...,No_Pool,Minimum_Privacy,None,0,3,2010,WD,Normal,-93.638933,42.060899


### Step 1 — Find the categorical features

1. Call `data.info()` in the cell below.
2. **Scan the output**:
   - Which columns are `category` or `object` (string) type?
   - Roughly how many categorical columns are there?
3. Use `data.select_dtypes` to keep only the categorical columns.

Before you run the code, ask yourself:

- **Prediction:** Do you think there are *a few* or *many* categorical columns in this dataset?
- How might that affect one-hot encoding?

In [2]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 2930 entries, 0 to 2929
Data columns (total 80 columns):
 #   Column              Non-Null Count  Dtype   
---  ------              --------------  -----   
 0   MS_SubClass         2930 non-null   category
 1   MS_Zoning           2930 non-null   category
 2   Lot_Frontage        2930 non-null   int64   
 3   Lot_Area            2930 non-null   int64   
 4   Street              2930 non-null   category
 5   Alley               2930 non-null   category
 6   Lot_Shape           2930 non-null   category
 7   Land_Contour        2930 non-null   category
 8   Utilities           2930 non-null   category
 9   Lot_Config          2930 non-null   category
 10  Land_Slope          2930 non-null   category
 11  Neighborhood        2930 non-null   category
 12  Condition_1         2930 non-null   category
 13  Condition_2         2930 non-null   category
 14  Bldg_Type           2930 non-null   category
 15  House_Style         2930 non-null   category
 16 

In [3]:
categorical_columns = data.select_dtypes(include=['str', 'category']).columns.tolist()
cat_data = data[categorical_columns]
cat_data.head()

,MS_SubClass,MS_Zoning,Street,Alley,Lot_Shape,Land_Contour,Utilities,Lot_Config,Land_Slope,Neighborhood,...,Garage_Type,Garage_Finish,Garage_Qual,Garage_Cond,Paved_Drive,Pool_QC,Fence,Misc_Feature,Sale_Type,Sale_Condition
0,One_Story_1946_and_Newer_All_Styles,Residential_Low_Density,Pave,No_Alley_Access,Slightly_Irregular,Lvl,AllPub,Corner,Gtl,North_Ames,...,Attchd,Fin,Typical,Typical,Partial_Pavement,No_Pool,No_Fence,None,WD,Normal
1,One_Story_1946_and_Newer_All_Styles,Residential_High_Density,Pave,No_Alley_Access,Regular,Lvl,AllPub,Inside,Gtl,North_Ames,...,Attchd,Unf,Typical,Typical,Paved,No_Pool,Minimum_Privacy,None,WD,Normal
2,One_Story_1946_and_Newer_All_Styles,Residential_Low_Density,Pave,No_Alley_Access,Slightly_Irregular,Lvl,AllPub,Corner,Gtl,North_Ames,...,Attchd,Unf,Typical,Typical,Paved,No_Pool,No_Fence,Gar2,WD,Normal
3,One_Story_1946_and_Newer_All_Styles,Residential_Low_Density,Pave,No_Alley_Access,Regular,Lvl,AllPub,Corner,Gtl,North_Ames,...,Attchd,Fin,Typical,Typical,Paved,No_Pool,No_Fence,None,WD,Normal
4,Two_Story_1946_and_Newer,Residential_Low_Density,Pave,No_Alley_Access,Slightly_Irregular,Lvl,AllPub,Inside,Gtl,Gilbert,...,Attchd,Fin,Typical,Typical,Paved,No_Pool,Minimum_Privacy,None,WD,Normal


### Step 2 — Quantify memory *before* and *after* one-hot encoding

In the next few cells you will:

1. Compute the **number of columns** and **memory usage** for the raw categorical data (`cat_data`).
2. Apply **`OneHotEncoder`** to those columns.
3. Compute the **number of columns** and **memory usage** for the encoded data.

As you work through the cells, keep track of:

- How many columns do you start with?
- How many columns do you end up with after encoding?
- How many **times larger** is the memory usage after encoding?

Try to write down your guesses before you run the code, and then compare them with the actual numbers.

In [4]:
ncols_before = cat_data.shape[1]
mem_before = cat_data.memory_usage(deep=True).sum() / 1024**2  # MB

In [5]:
encoder = OneHotEncoder(
    sparse_output=False,
    handle_unknown='ignore',
)

encoder.set_output(transform='pandas')

cat_data_encoded = encoder.fit_transform(data[categorical_columns])

In [6]:
ncols_after = cat_data_encoded.shape[1]
mem_after = cat_data_encoded.memory_usage(deep=True).sum() / 1024**2  # MB

### Step 3 — Reflect on the impact of one-hot encoding

Run the cell below to print the **number of columns** and **memory usage** before and after one-hot encoding.

Then answer these questions in your own words (e.g., in a separate markdown cell or your notes):

- By what **factor** did the number of columns increase?
- By what **factor** did the memory usage increase?
- Why might this be a problem for datasets with even **higher cardinality** (more unique categories) or many more rows?
- How could parameters like `min_frequency` or `max_categories` in `OneHotEncoder` help mitigate this issue?

Finally, compare your results to the summary in the `lab_encode` notebook. Do they roughly match what you saw there?

In [7]:
print(f"Columns Before: {ncols_before}")
print(f"Columns After: {ncols_after}")
print(f"Memory Before: {mem_before:.2f} MB")
print(f"Memory After: {mem_after:.2f} MB")

Columns Before: 46
Columns After: 318
Memory Before: 0.15 MB
Memory After: 7.11 MB
